In [1]:
# 全局设置
import os
import warnings
warnings.filterwarnings('ignore')
import datetime as dt

import numpy as np
import pandas as pd
from IPython.display import Markdown

from QuantStudio.Tools.Visualization import qs_help

# 多因子风险库

多因子风险库 `HDF5FRDB` 是专门用于存储结构化多因子风险数据的数据库实现，底层基于 HDF5 文件。

In [2]:
# 创建多因子风险库并连接
from QuantStudio.Risk.HDF5RDB import HDF5FRDB

RDB = HDF5FRDB(args={"MainDir": "../data/Risk"}).connect()

print(qs_help(RDB))

2026-06-27 20:52:32,403 | QS | WARNING : 找不到配置文件: C:\Users\hst\QuantStudioConfig\HDF5FRDBConfig.json


类型: HDF5FRDB
模块: QuantStudio.Risk.HDF5RDB
QS 对象类型: 风险库
QS 对象名称: HDF5FRDB
QSID: 0a45d3c8b3cbbe6ef7b670b89ec6b3b7d22246fc992c1968927ae4614b4385c0
参数集:
    * Name(名称): <class 'str'>, 默认值 'HDF5FRDB', 当前取值: 'HDF5FRDB'
    * MainDir(主目录): <class 'pathlib.Path'>, 无默认值, 存放数据的主目录, 当前取值: ..\data\Risk
说明文档:
    基于 HDF5 文件的多因子风险数据库


In [3]:
# 风险库的参数集
display(Markdown(RDB.Args.info()))

* Name(名称): <class 'str'>, 默认值 'HDF5FRDB', 当前取值: 'HDF5FRDB'
* MainDir(主目录): <class 'pathlib.Path'>, 无默认值, 存放数据的主目录, 当前取值: ..\data\Risk

## 风险表列表

In [4]:
# 风险库中的所有风险表列表
RDB.TableNames

['demo_risk_table']

## 无结构风险库

QuantStudio 也提供了不分解风险结构的普通风险库 `HDF5RDB`，用于存储直接的协方差矩阵（不分解为因子模型）。其 API 与 `HDF5FRDB` 基本一致，区别在于只操作 `icov`（协方差矩阵）而不涉及因子维度的数据。

```python
from QuantStudio.Risk.HDF5RDB import HDF5RDB

# 创建普通风险库
SimpleRDB = HDF5RDB(args={"MainDir": "../data/Risk"}).connect()
print(SimpleRDB.TableNames)

# 写入普通风险数据（单个时点的协方差矩阵）
import pandas as pd
icov = pd.DataFrame([[0.04, 0.01], [0.01, 0.09]],
                     index=["000001.SZ", "000002.SZ"],
                     columns=["000001.SZ", "000002.SZ"])
SimpleRDB.writeData(table_name="simple_risk_table", idt=dt.datetime(2025, 6, 1), icov=icov)

# 读取
SimpleRT = SimpleRDB.getTable("simple_risk_table")
print(SimpleRT.readCov(dts=[dt.datetime(2025, 6, 1)]))

# 清理
SimpleRDB.deleteTable("simple_risk_table")
```

# 风险表

## 获取风险表对象

In [5]:
# 获取风险库中的某个风险表对象
RT = RDB.getTable("demo_risk_table")
print(qs_help(RT))

类型: HDF5FactorRiskTable
模块: QuantStudio.Risk.HDF5RDB
QS 对象类型: 计算节点-风险表
QS 对象名称: demo_risk_table
QSID: 41ed0116445fc6e0b76ac83224a576452f85c550d546dbe5094d1dc6748e18a8
参数集:
    * Name: <class 'str'>, 默认值 '风险表', 当前取值: 'demo_risk_table'
说明文档:
    基于 HDF5 文件的多因子风险表


In [6]:
# 风险表的参数集
display(Markdown(RT.Args.info()))

* Name: <class 'str'>, 默认值 '风险表', 当前取值: 'demo_risk_table'

## 元信息

In [7]:
# 获取风险表元信息的方法
print(qs_help(RT.getMetaData))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FactorRiskTable.getMetaData(key: Optional[str] = None) -> Union[pandas.core.series.Series, Any]
说明文档:
    获取风险表的元信息, 元信息由若干个键值对组成
    
    Args:
        key: 元信息键, None 表示获取所有的元信息
    
    Returns:
        如果 key 非 None 则返回该 key 对应的元信息
        如果 key=None, 则返回 Series(index=[所有的 key])


In [8]:
# 读取风险表的所有元信息
print(RT.getMetaData())

Description    这是一张示例风险表
dtype: object


In [9]:
# 读取风险表的某个元信息
print(RT.getMetaData(key="Description"))

这是一张示例风险表


## 时点序列

getDateTime(start_dt=None, end_dt=None):
* start_dt: datetime 或者 None, 起始时点
* end_dt: datetime 或者 None, 终止时点
* 返回: list(datetime)

In [10]:
# 获取时点序列的方法
print(qs_help(RT.getDateTime))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FactorRiskTable.getDateTime(start_dt: Optional[datetime.datetime] = None, end_dt: Optional[datetime.datetime] = None) -> List[datetime.datetime]
说明文档:
    获取时点序列
    
    Args:
        start_dt: 起始日, 非 None 表示截取 start_dt 之后的时点
        end_dt: 结束日, 非 None 表示截取 end_dt 之前的时点
    
    Returns:
        时点序列, 若为空 list, 表示该风险表没有固定的时点序列或者无法获取


In [11]:
# 获取风险表的时点序列
DTs = RT.getDateTime()
print(DTs[0], " - ", DTs[-1])

2025-01-01 00:00:00  -  2025-04-30 00:00:00


In [12]:
# 给定起始时点和截止时点, 获取风险表的时点序列
RT.getDateTime(start_dt=dt.datetime(2025, 1, 5), end_dt=dt.datetime(2025, 1, 8))

[datetime.datetime(2025, 1, 5, 0, 0),
 datetime.datetime(2025, 1, 6, 0, 0),
 datetime.datetime(2025, 1, 7, 0, 0),
 datetime.datetime(2025, 1, 8, 0, 0)]

## ID 序列

In [13]:
# 获取风险表 ID 序列的方法
print(qs_help(RT.getID))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FactorRiskTable.getID(idt: Optional[datetime.datetime] = None) -> List[str]
说明文档:
    获取 ID 序列
    
    Args:
        idt: 给定的时点, 非 None 表示获取该时点的 ID 序列, None 表示获取所有的 ID 序列
    
    Returns:
        ID 序列, 若为空 list, 表示该风险表没有固定的 ID 序列或者无法获取


In [14]:
# 获取风险表的 ID 序列
IDs = RT.getID()
print(IDs[0], ", ..., ", IDs[-1])

000001.SZ , ...,  000020.SZ


In [15]:
# 给定目标时点, 获取风险表中指定时点的 ID 序列
IDs = RT.getID(idt=dt.datetime(2025, 1, 5))
print(IDs[0], ", ..., ", IDs[-1])

000001.SZ , ...,  000020.SZ


## 读取数据

In [16]:
# 风险表数据读取方法
print(qs_help(RT.readCov))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.RiskTable
签名: FactorRT.readCov(dts: List[datetime.datetime], ids: Optional[List[str]] = None) -> QuantStudio.Core.QSObject.Panel
说明文档:
    读取风险矩阵
    
    Args:
        dts: 时点序列
        ids: ID 序列, None 表示读取所有的 ID
    
    Returns:
        Panel(item=dts, major_axis=ids, minor_axis=ids)


In [17]:
# 给定时点列表, ID 列表, 获取风险数据
DTs = RT.getDateTime()
Data = RT.readCov(dts=DTs, ids=["000001.SZ", "000002.SZ", "000003.SZ", "000990.SZ"])
print("三维数据 : ")
print(Data)

iDT = dt.datetime(2025, 1, 5)
print(f"时点切片 : {iDT}")
print(Data.loc[iDT])

iID = "000001.SZ"
print(f"ID 切片 : {iID}")
print(Data.loc[:, iID, iID])

三维数据 : 
<class 'QuantStudio.Core.QSObject.Panel'>
Dimensions: 120 (items) x 4 (major_axis) x 4 (minor_axis)
Items axis: 2025-01-01 00:00:00 to 2025-04-30 00:00:00
Major_axis axis: 000001.SZ to 000990.SZ
Minor_axis axis: 000001.SZ to 000990.SZ
时点切片 : 2025-01-05 00:00:00
           000001.SZ  000002.SZ  000003.SZ  000990.SZ
000001.SZ   3.269278  -0.286344  -0.005609        NaN
000002.SZ  -0.286344   4.494141   1.590149        NaN
000003.SZ  -0.005609   1.590149  12.908544        NaN
000990.SZ        NaN        NaN        NaN        NaN
ID 切片 : 000001.SZ
2025-01-01     3.408532
2025-01-02    11.536865
2025-01-03     4.853806
2025-01-04     4.015310
2025-01-05     3.269278
                ...    
2025-04-26     7.307067
2025-04-27     2.383721
2025-04-28     6.737270
2025-04-29    13.331130
2025-04-30     7.366742
Length: 120, dtype: float64


# 多因子风险表

`HDF5FactorRiskTable` 是多因子风险表，除了继承 `RiskTable` 的所有方法外，还提供因子维度的数据读取功能。其存储的数据模型为：

$$\mathbf{V} = \mathbf{X} \cdot \mathbf{F} \cdot \mathbf{X}^T + \mathbf{\Delta}$$

## 因子列表

In [18]:
# 获取风险表中的所有因子列表
print(RT.FactorNames)

['Beta', 'Momentum', 'NonlinearSize', 'ResidualVolatility', 'Size']


## 时点序列

多因子风险表提供三种时点序列：
- `getDateTime()`：风险矩阵的时点（因子协方差阵和特异性风险的时点）
- `getFactorReturnDateTime()`：因子收益率的时点
- `getSpecificReturnDateTime()`：特异性收益率的时点

### 因子收益时点序列

In [19]:
# 风险表因子收益时点序列读取方法
print(qs_help(RT.getFactorReturnDateTime))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FactorRiskTable.getFactorReturnDateTime(start_dt: Optional[datetime.datetime] = None, end_dt: Optional[datetime.datetime] = None) -> List[datetime.datetime]
说明文档:
    获取因子收益的时点序列
    
    Args:
        start_dt: 起始日, 非 None 表示截取 start_dt 之后的时点
        end_dt: 结束日, 非 None 表示截取 end_dt 之前的时点
    
    Returns:
        时点序列, 若为空 list, 表示该风险表没有固定的因子收益时点序列或者无法获取


In [20]:
# 获取风险表因子收益的时点序列
DTs = RT.getFactorReturnDateTime(dt.datetime(2025, 1, 5), end_dt=dt.datetime(2025, 1, 8))
print(DTs[0], " - ", DTs[-1])

2025-01-05 00:00:00  -  2025-01-08 00:00:00


### 特异性收益时点序列

In [21]:
# 风险表特异性收益时点序列读取方法
print(qs_help(RT.getSpecificReturnDateTime))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FactorRiskTable.getSpecificReturnDateTime(start_dt: Optional[datetime.datetime] = None, end_dt: Optional[datetime.datetime] = None) -> List[datetime.datetime]
说明文档:
    获取特异性收益的时点序列
    
    Args:
        start_dt: 起始日, 非 None 表示截取 start_dt 之后的时点
        end_dt: 结束日, 非 None 表示截取 end_dt 之前的时点
    
    Returns:
        时点序列, 若为空 list, 表示该风险表没有固定的特异性收益时点序列或者无法获取


In [22]:
# 获取风险表特异性收益的时点序列
DTs = RT.getSpecificReturnDateTime(dt.datetime(2025, 1, 5), end_dt=dt.datetime(2025, 1, 8))
print(DTs[0], " - ", DTs[-1])

2025-01-05 00:00:00  -  2025-01-08 00:00:00


## 读取数据

多因子风险表通过以下方法读取结构化风险的各个组件。各组件间的数学关系为 $\mathbf{V} = \mathbf{X} \cdot \mathbf{F} \cdot \mathbf{X}^T + \mathbf{\Delta}$。

### 因子协方差阵

因子协方差矩阵 $\mathbf{F}$，维度为 $K \times K$（$K$ 为因子数），通过 `readFactorCov` 读取。

In [23]:
# 因子协方差矩阵读取方法
print(qs_help(RT.readFactorCov))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FactorRiskTable.readFactorCov(dts: List[datetime.datetime]) -> QuantStudio.Core.QSObject.Panel
说明文档:
    读取因子风险矩阵
    
    Args:
        dts: 时点序列
    
    Returns:
        Panel(item=dts, major_axis=[因子], minor_axis=[因子])


In [24]:
# 给定时点列表, 获取因子风险数据
DTs = RT.getDateTime()
Data = RT.readFactorCov(dts=DTs)
print("三维数据 : ")
print(Data)

iDT = dt.datetime(2025, 1, 5)
print(f"时点切片 : {iDT}")
print(Data.loc[iDT].iloc[:3, :3])

三维数据 : 
<class 'QuantStudio.Core.QSObject.Panel'>
Dimensions: 120 (items) x 5 (major_axis) x 5 (minor_axis)
Items axis: 2025-01-01 00:00:00 to 2025-04-30 00:00:00
Major_axis axis: Size to NonlinearSize
Minor_axis axis: Size to NonlinearSize
时点切片 : 2025-01-05 00:00:00
              Size      Beta  Momentum
Size      0.876821  0.041411 -0.096047
Beta      0.041411  0.860359 -0.070212
Momentum -0.096047 -0.070212  1.031065


### 特异性风险

In [25]:
# 特异性风险读取方法
print(qs_help(RT.readSpecificRisk))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FactorRiskTable.readSpecificRisk(dts: List[datetime.datetime], ids: Optional[List[str]] = None) -> pandas.core.frame.DataFrame
说明文档:
    读取特异性风险
    
    Args:
        dts: 时点序列
        ids: 证券 ID 序列, None 表示获取表里所有的 ID
    
    Returns:
        DataFrame(index=dts, columns=ids)


In [26]:
# 读取特异性风险
DTs = RT.getDateTime()
Data = RT.readSpecificRisk(dts=DTs)
print(Data.iloc[:5, :3])

            000001.SZ  000002.SZ  000003.SZ
2025-01-01   0.311875   0.515763   0.121525
2025-01-02   0.320684   0.043222   0.080116
2025-01-03   0.374758   0.190959   0.150083
2025-01-04   0.532308   0.248100   0.173775
2025-01-05   0.043449   0.927212   0.825611


### 因子暴露

因子暴露矩阵 $\mathbf{X}$，维度为 $N \times K$（$N$ 为证券数），通过 `readFactorData` 读取。返回的 Panel 维度为 `items=[因子], major_axis=dts, minor_axis=ids`。

In [27]:
# 因子暴露数据读取方法
print(qs_help(RT.readFactorData))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FactorRiskTable.readFactorData(dts: List[datetime.datetime], ids: Optional[List[str]] = None) -> QuantStudio.Core.QSObject.Panel
说明文档:
    读取因子截面数据
    
    Args:
        dts: 时点序列
        ids: 证券 ID 序列, None 表示获取表里所有的 ID
    
    Returns:
        Panel(items=[因子], major_axis=dts, minor_axis=ids)


In [28]:
# 读取截面因子数据
DTs = RT.getDateTime()
Data = RT.readFactorData(dts=DTs)
print("三维数据 : ")
print(Data)

iDT = dt.datetime(2025, 1, 5)
print(f"时点切片 : {iDT}")
print(Data.loc[:, iDT].iloc[:5, :3])

三维数据 : 
<class 'QuantStudio.Core.QSObject.Panel'>
Dimensions: 5 (items) x 120 (major_axis) x 20 (minor_axis)
Items axis: Size to NonlinearSize
Major_axis axis: 2025-01-01 00:00:00 to 2025-04-30 00:00:00
Minor_axis axis: 000001.SZ to 000020.SZ
时点切片 : 2025-01-05 00:00:00
               Size      Beta  Momentum
000001.SZ -1.165150  0.634387 -1.427675
000002.SZ  0.900826  1.629276  0.352934
000003.SZ  0.465662  0.139064 -0.294159
000004.SZ -1.536244 -0.857670 -0.956312
000005.SZ  1.488252 -1.249339 -1.345773


### 因子收益率

In [29]:
# 因子收益率数据读取方法
print(qs_help(RT.readFactorReturn))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FactorRiskTable.readFactorReturn(dts: List[datetime.datetime]) -> pandas.core.frame.DataFrame
说明文档:
    读取因子收益率
    
    Args:
        dts: 时点序列
    
    Returns:
        DataFrame(index=dts, columns=[因子])


In [30]:
# 读取因子收益率
DTs = RT.getFactorReturnDateTime()
Data = RT.readFactorReturn(dts=DTs)
print(Data.iloc[:5, :3])

                Size      Beta  Momentum
2025-01-01  0.696520  0.184530 -0.789265
2025-01-02  0.165060 -1.611395  0.259272
2025-01-03 -0.801664  0.687669  0.784019
2025-01-04  1.220110  0.580694 -0.896004
2025-01-05 -0.967076  0.324836 -0.492945


### 特异性收益率

In [31]:
# 特异性收益率数据读取方法
print(qs_help(RT.readSpecificReturn))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FactorRiskTable.readSpecificReturn(dts: List[datetime.datetime], ids: Optional[List[str]] = None) -> pandas.core.frame.DataFrame
说明文档:
    读取特异性收益率
    
    Args:
        dts: 时点序列
        ids: 证券 ID 序列, None 表示获取表里所有的 ID
    
    Returns:
        DataFrame(index=dts, columns=ids)


In [32]:
# 读取特异性收益率
DTs = RT.getSpecificReturnDateTime()
Data = RT.readSpecificReturn(dts=DTs)
print(Data.iloc[:5, :3])

            000001.SZ  000002.SZ  000003.SZ
2025-01-01   1.025299  -2.051276   1.009152
2025-01-02  -1.018024  -0.660692   1.651123
2025-01-03   0.145802  -1.131834  -0.296888
2025-01-04  -0.765229   0.141932   0.949451
2025-01-05  -0.293964  -1.318126  -0.703258


# 数据写入

`HDF5FRDB.writeData` 支持逐时点写入多因子风险数据的各个组件，各参数均为可选——可以只写入部分数据（如只写入因子收益率而不写因子协方差阵）。

### 通用数据读取

除标准的多因子风险数据组件外，`readData` 方法可以读取表中存储的任意数据项（如市值 `Cap`、回归统计量 `Statistics` 等）。

```python
# 读取回归统计量
Statistics = RT.readData("Statistics", dts=RT.getDateTime())
if Statistics is not None:
    print(Statistics.iloc[0])
```

## 写入多因子风险数据

In [33]:
# HDF5FRDB 写入方法
print(qs_help(RDB.writeData))

类型: method (bound to HDF5FRDB)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FRDB.writeData(table_name: str, idt: datetime.datetime, factor_data: Optional[pandas.core.frame.DataFrame] = None, factor_cov: Optional[pandas.core.frame.DataFrame] = None, specific_risk: Optional[pandas.core.series.Series] = None, factor_ret: Optional[pandas.core.series.Series] = None, specific_ret: Optional[pandas.core.series.Series] = None, **kwargs)
说明文档:
    写入风险数据
    
    Args:
        table_name: 风险表名称
        idt: 待写入的时点
        factor_data: 因子截面数据, DataFrame(index=[ID], columns=[因子]), 其中 index 是证券代码, columns 是因子列表
        factor_cov: 因子协方差矩阵, DataFrame(index=[因子], columns=[因子]), 其中 index 和 columns 都是因子列表
        specific_risk: 特异性风险: Series(index=[ID]), 其中 index 是证券代码
        factor_ret: 因子收益率, Series(index=[因子]), 其中 index 是因子列表
        specific_ret: 特异性收益率, Series(index=[ID]), 其中 index 是证券代码


### 写入示例

In [34]:
# 数据写入示例：从 demo_risk_table 读取数据，写入新表 TestTable
RT = RDB.getTable("demo_risk_table")
CovDTs = RT.getDateTime()[-2:]  # 取最后两个时点（有因子协方差阵的时点）
IDs = RT.getID()[:5]

# 读取各组件数据
FactorCov = RT.readFactorCov(dts=CovDTs)
SpecificRisk = RT.readSpecificRisk(dts=CovDTs, ids=IDs)

# 取前 10 个因子收益时点（收益数据时点范围通常比风险矩阵更广）
DTs = RT.getFactorReturnDateTime(start_dt=CovDTs[0]-dt.timedelta(31), end_dt=CovDTs[-1])[:10]
FactorData = RT.readFactorData(dts=DTs, ids=IDs)
FactorReturn = RT.readFactorReturn(dts=DTs)
SpecificReturn = RT.readSpecificReturn(dts=DTs, ids=IDs)

# 逐时点写入
for iDT in DTs:
    if iDT in CovDTs:
        # 风险矩阵时点：写入全量数据
        RDB.writeData(table_name="TestTable", idt=iDT,
                      factor_data=FactorData.loc[:, iDT],
                      factor_cov=FactorCov.loc[iDT],
                      specific_risk=SpecificRisk.loc[iDT],
                      factor_ret=FactorReturn.loc[iDT],
                      specific_ret=SpecificReturn.loc[iDT])
    else:
        # 仅收益数据时点：只写入因子收益率和特异性收益率
        RDB.writeData(table_name="TestTable", idt=iDT,
                      factor_data=FactorData.loc[:, iDT],
                      factor_cov=None,
                      specific_risk=None,
                      factor_ret=FactorReturn.loc[iDT],
                      specific_ret=SpecificReturn.loc[iDT])

print("写入完成，当前风险表列表:", RDB.TableNames)

写入完成，当前风险表列表: ['TestTable', 'demo_risk_table']


In [35]:
# 验证写入结果：读取新表的因子收益率
NewRT = RDB.getTable("TestTable")
print("新表的因子收益率时点序列:", NewRT.getFactorReturnDateTime())
print("新表的时点序列（有风险矩阵的时点）:", NewRT.getDateTime())
print("新表的因子列表:", NewRT.FactorNames)

新表的因子收益率时点序列: [datetime.datetime(2025, 3, 29, 0, 0), datetime.datetime(2025, 3, 30, 0, 0), datetime.datetime(2025, 3, 31, 0, 0), datetime.datetime(2025, 4, 1, 0, 0), datetime.datetime(2025, 4, 2, 0, 0), datetime.datetime(2025, 4, 3, 0, 0), datetime.datetime(2025, 4, 4, 0, 0), datetime.datetime(2025, 4, 5, 0, 0), datetime.datetime(2025, 4, 6, 0, 0), datetime.datetime(2025, 4, 7, 0, 0)]
新表的时点序列（有风险矩阵的时点）: [datetime.datetime(2025, 3, 29, 0, 0), datetime.datetime(2025, 3, 30, 0, 0), datetime.datetime(2025, 3, 31, 0, 0), datetime.datetime(2025, 4, 1, 0, 0), datetime.datetime(2025, 4, 2, 0, 0), datetime.datetime(2025, 4, 3, 0, 0), datetime.datetime(2025, 4, 4, 0, 0), datetime.datetime(2025, 4, 5, 0, 0), datetime.datetime(2025, 4, 6, 0, 0), datetime.datetime(2025, 4, 7, 0, 0)]
新表的因子列表: []


# 风险表管理操作

除了基本的读写操作外，`HDF5FRDB` 还支持对风险表进行管理：设置元信息、重命名、删除整表或删除指定时点的数据。

## 删除指定时点数据

```python
# 删除表中某个时点的数据
# RDB.deleteDateTime(table_name="TestTable", dts=[dt.datetime(2025, 1, 1)])
```

## 设置表的元信息

In [36]:
# 设置表元信息方法
print(qs_help(RDB.setTableMetaData))

类型: method (bound to HDF5FRDB)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FRDB.setTableMetaData(table_name: str, key: Optional[str] = None, value: Any = None, meta_data: Optional[dict] = None)
说明文档:
    设置风险表的元信息, 元信息由若干个键值对组成
    
    Args:
        table_name: 风险表名称
        key: 元信息键
        value: 元信息值
        meta_data: 若干组键值对元信息


In [37]:
# 设置表的元信息
TargetTable = "TestTable"
RT = RDB.getTable(TargetTable)
print("设置前的元信息 : ")
print(RT.getMetaData())
RDB.setTableMetaData(table_name=TargetTable, key="Description", value="这是一张测试表")
print("设置后的元信息 : ")
print(RT.getMetaData())

设置前的元信息 : 
Series([], dtype: object)
设置后的元信息 : 
Description    这是一张测试表
dtype: object


## 重命名表

In [38]:
# 重命名表方法
print(qs_help(RDB.renameTable))

类型: method (bound to HDF5FRDB)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FRDB.renameTable(old_table_name: str, new_table_name: str)
说明文档:
    重命名表
    
    Args:
        old_table_name: 原表名
        new_table_name: 新表名


In [39]:
# 重命名表
print("重命名前风险表 : ")
print(RDB.TableNames)
RDB.renameTable(old_table_name="TestTable", new_table_name="TestTable_New")
print("重命名后风险表 : ")
print(RDB.TableNames)

重命名前风险表 : 
['TestTable', 'demo_risk_table']
重命名后风险表 : 
['TestTable_New', 'demo_risk_table']


In [40]:
# 删除表
print("删除前风险表 : ")
print(RDB.TableNames)
RDB.deleteTable(table_name="TestTable_New")
print("删除后风险表 : ")
print(RDB.TableNames)

删除前风险表 : 
['TestTable_New', 'demo_risk_table']
删除后风险表 : 
['demo_risk_table']
